# **OBI-WAN**: **O**ccupation **B**ased **I**ndex for **W**orkforce **A**I **N**avigator

# Notebook Roadmap

## OBI-WAN: AI Career Navigator Prototype
**Architecture & Execution Environment**

This notebook contains the functional prototype for OBI-WAN. It is organized into the following architectural stages:

1. **Environment Setup & Authentication:** Import core libraries, authenticate via Google Colab OAuth for secure BigQuery data access, and load the Gemini API key from the Colab secrets vault to instantiate the Generative AI client.

2. **Data Pipeline & Vector Retrieval:** Establish the BigQuery client and load high-dimensional embedding tables for semantic routing.

3. **Tool Execution Logic:** Define the deterministic Python functions (Tools) that query BigQuery for skills, career forecasting, and institution mapping.

4. **Agent Architecture:** Instantiate the core LLM agent, injecting system instructions and wiring the custom BigQuery tools to the model's reasoning loop.

5. **Prototype Interface (ADK Web UI):** Launch the agent into an interactive, local web application using the Google ADK.

6. **Prototype Demonstration**

7. **Adversarial Safety Evaluation:** (See the `evaluation/` folder in this repository for the synthetic stress-testing framework designed to validate this AI's logic).

# 1. Environment Setup & Authentication

In [ ]:
# --- INSTALLATIONS ---
!pip install google-cloud-bigquery google-generativeai

# --- CORE IMPORTS ---
import os
import numpy as np
import pandas as pd
from typing import Optional

# --- GOOGLE COLAB & CLOUD IMPORTS ---
from google.colab import auth
from google.colab import userdata
from google.cloud import bigquery

# --- AI & ADK IMPORTS ---
import google.generativeai as genai
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini

# --- BASIC CONFIG ---
GCP_PROJECT_ID = "dashboard-458701"
REGION = "us-west1"

# --- INITIALIZATION ---
def initialize_services():
    try:
        # 1. Cloud Authentication
        print("⏳ Waiting for Google Cloud authentication popup...")
        auth.authenticate_user()

        # 2. API Key Management
        google_api_key = userdata.get("GOOGLE_API_KEY")
        os.environ["GOOGLE_API_KEY"] = google_api_key
        genai.configure(api_key=google_api_key)

        # 3. Client Creation
        bq_client = bigquery.Client(project=GCP_PROJECT_ID, location=REGION)

        print("Environment successfully authenticated.")
        return bq_client, google_api_key

    except Exception as e:
        print(f"Initialization Error: {e}")
        return None, None

# Execute
bq_client, GOOGLE_API_KEY = initialize_services()


# 2. Data Pipeline & Vector Retrieval

To minimize latency and cloud compute costs, this section establishes a caching layer and a custom Retrieval-Augmented Generation (RAG) pipeline.
* **Ingestion:** `load_job_registry` and `load_program_registry` query BigQuery and cache the vector spaces as NumPy arrays.
* **Retrieval Math:** `_cosine_sims` performs the core similarity calculations between the user's embedded prompt and the cached BigQuery data to surface the most contextually relevant educational pathways.

In [ ]:
### The Configuration & Caching Layer

# ------------------------------------------------------------------
# --- CONFIGURATION ---
# ------------------------------------------------------------------
GCP_PROJECT_ID = "dashboard-458701"
REGION = "us-west1"

JOB_EMBED_TABLE_ID = "onet.job_embeddings_v2"
PROGRAM_EMBED_TABLE_ID = "ipeds.program_embeddings"

NODE_OCCUPATIONS = "onet.occupations_node"
NODE_COMPETENCIES = "onet.competencies_node"
NODE_PROGRAMS = "ipeds.programs_node"
NODE_INSTITUTIONS = "ipeds.institutions_node"

EDGE_JOB_COMPETENCY = "onet.job_requires_competencies_edge"
EDGE_PROGRAM_JOB = "crosswalk.program_leads_to_job_edge"
EDGE_INST_PROGRAM = "ipeds.institution_offers_programs_edge"

AI_SCORE_TABLE_ID = "onet.final_ai_applicability_multiplicative"
AI_SCORE_COL = "AI_Applicability_Score"

EMBED_MODEL_NAME = "text-embedding-004"

# ------------------------------------------------------------------
# --- CACHING GLOBALS ---
# ------------------------------------------------------------------
job_registry_df = None
job_emb_matrix = None

program_registry_df = None
program_emb_matrix = None



### The Ingestion Engine

def load_job_registry():
    global job_registry_df, job_emb_matrix
    if not bq_client:
        raise RuntimeError("BigQuery client not initialized.")

    if job_registry_df is None:
        print("⏳ Loading Job Vector Registry...")
        sql = f"""
            SELECT soc_code AS onet_soc_code, title AS Title, embedding
            FROM `{GCP_PROJECT_ID}.{JOB_EMBED_TABLE_ID}`
            WHERE embedding IS NOT NULL
        """
        job_registry_df = bq_client.query(sql).to_dataframe()
        job_emb_matrix = np.array(job_registry_df["embedding"].tolist(), dtype=np.float32)

        if job_emb_matrix.size > 0:
            print(
                f"Job Registry Loaded: {len(job_registry_df)} jobs (dim={job_emb_matrix.shape[1]})."
            )
        else:
            print("Job Registry loaded, but embedding matrix is empty.")


def load_program_registry():
    global program_registry_df, program_emb_matrix
    if not bq_client:
        raise RuntimeError("BigQuery client not initialized.")

    if program_registry_df is None:
        print("⏳ Loading Program Vector Registry...")
        sql = f"""
            SELECT cip_code, title, embedding
            FROM `{GCP_PROJECT_ID}.{PROGRAM_EMBED_TABLE_ID}`
            WHERE embedding IS NOT NULL
        """
        program_registry_df = bq_client.query(sql).to_dataframe()
        program_emb_matrix = np.array(
            program_registry_df["embedding"].tolist(), dtype=np.float32
        )

        if program_emb_matrix.size > 0:
            print(
                f"Program Registry Loaded: {len(program_registry_df)} programs (dim={program_emb_matrix.shape[1]})."
            )
        else:
            print("Program Registry loaded, but embedding matrix is empty.")



### The Mathematics (The Core RAG Engine)

def _embed_query_vertex(user_query: str) -> np.ndarray:
    if embed_model is None:
        raise RuntimeError("Embedding model not initialized.")
    if user_query is None or str(user_query).strip() == "":
        raise ValueError("User query is empty; cannot embed.")

    inputs = [TextEmbeddingInput(str(user_query), task_type="RETRIEVAL_QUERY")]
    result = embed_model.get_embeddings(inputs)
    return np.array(result[0].values, dtype=np.float32)


def _cosine_sims(matrix: np.ndarray, q_vec: np.ndarray) -> np.ndarray:
    denom = (np.linalg.norm(matrix, axis=1) * np.linalg.norm(q_vec) + 1e-8)
    return np.dot(matrix, q_vec) / denom


### The Execution Layer

def find_topk_nodes_via_vector(
    user_query: str, type: str = "program", k: int = 3
) -> List[Dict[str, object]]:
    if type == "job":
        load_job_registry()
        df, matrix, id_col, title_col = (
            job_registry_df,
            job_emb_matrix,
            "onet_soc_code",
            "Title",
        )
    else:
        load_program_registry()
        df, matrix, id_col, title_col = (
            program_registry_df,
            program_emb_matrix,
            "cip_code",
            "title",
        )

    if df is None or df.empty or matrix is None or matrix.size == 0:
        return []

    q_vec = _embed_query_vertex(user_query)

    if matrix.shape[1] != q_vec.shape[0]:
        raise ValueError(
            f"Embedding dimension mismatch for type='{type}'. "
            f"Registry dim={matrix.shape[1]}, query dim={q_vec.shape[0]}."
        )

    sims = _cosine_sims(matrix, q_vec)
    k = max(1, int(k))
    k = min(k, len(sims))

    top_idx = np.argpartition(-sims, kth=k - 1)[:k]
    top_idx = top_idx[np.argsort(-sims[top_idx])]

    out: List[Dict[str, object]] = []
    for rank, idx in enumerate(top_idx, start=1):
        row = df.iloc[int(idx)]
        out.append(
            {
                "id": row[id_col],
                "title": row[title_col],
                "score": float(sims[int(idx)]),
                "rank": rank,
            }
        )

    if os.getenv("OBIWAN_DEBUG", "0") == "1":
        print(f"---- OBIWAN DEBUG: find_topk_nodes_via_vector(type={type}) ----")
        print(f"Query: {user_query}")
        for m in out:
            print(
                f"  #{m['rank']}: {m['title']} (ID={m['id']}) score={m['score']:.4f}"
            )
        print("--------------------------------------------------------------")

    return out


def find_node_via_vector(user_query: str, type: str = "job") -> Tuple[Optional[str], Optional[str]]:
    matches = find_topk_nodes_via_vector(user_query, type=type, k=1)
    if not matches:
        return None, None
    return str(matches[0]["id"]), str(matches[0]["title"])

# 3. Tool Execution Logic

In [ ]:
# ------------------------------------------------------------------
# NORMALIZERS
# ------------------------------------------------------------------
def normalize_comp_category(category: Optional[str]) -> str:
    c = (category or "Skill").strip().lower()
    if "know" in c:
        return "Knowledge"
    if "abil" in c:
        return "Ability"
    return "Skill"

def normalize_award_level(user_text: Optional[str]) -> Optional[str]:
    if not user_text:
        return None
    t = user_text.strip().lower()
    if "associate" in t:
        return "associate"
    if "bachelor" in t or "undergrad degree" in t:
        return "bachelor"
    if "master" in t:
        return "master"
    if "doctor" in t or "phd" in t:
        return "doctor"
    if "graduate certificate" in t or "grad certificate" in t:
        return "grad_cert"
    if "undergraduate certificate" in t or "undergrad certificate" in t:
        return "ug_cert"
    if "certificate" in t:
        return "certificate"
    return None

def normalize_modality(user_text: Optional[str]) -> Optional[str]:
    if not user_text:
        return None
    t = user_text.strip().lower()
    if "online" in t or "remote" in t or "virtual" in t:
        return "DE ENTIRELY"
    if "hybrid" in t or "some" in t or "mixed" in t:
        return "DE SOME"
    if "on-campus" in t or "campus" in t or "person" in t or "f2f" in t:
        return "F2F"
    return None

def _award_sql_filters(aw_norm: Optional[str]) -> List[str]:
    if not aw_norm:
        return []
    if aw_norm == "associate":
        return ['e.award_level = "Associate\'s degree"']
    if aw_norm == "bachelor":
        return ['e.award_level = "Bachelor"']
    if aw_norm == "master":
        return ['e.award_level = "Master\'s degree"']
    if aw_norm == "doctor":
        return ["e.award_level LIKE 'Doctor%'", "e.award_level LIKE 'Doctoral%'"]
    if aw_norm == "ug_cert":
        return ["e.award_level LIKE 'Certificates of%'"]
    if aw_norm == "grad_cert":
        return [
            'e.award_level = "Post-master\'s certificate"',
            'e.award_level = "Postbaccalaureate certificate"',
        ]
    return []




# ------------------------------------------------------------------
# TOOL 1: Competencies (Skills, Knowledge, Abilities) Extraction per Occupation
# ------------------------------------------------------------------
def get_job_requirements(
    tool_context: ToolContext, job_title: str, category: str = "Skill", **kwargs
) -> str:
    if not bq_client:
        return "Error: BigQuery not ready."

    onet_code, official_title = find_node_via_vector(job_title, type="job")
    if not onet_code:
        return f"Could not find a match for '{job_title}'."

    tool_context.state["last_job_title"] = official_title
    ctype = normalize_comp_category(category)

    sql = f"""
        SELECT c.`Element Name` AS item_name, e.importance
        FROM `{GCP_PROJECT_ID}.{EDGE_JOB_COMPETENCY}` e
        JOIN `{GCP_PROJECT_ID}.{NODE_COMPETENCIES}` c
          ON TRIM(e.element_id) = TRIM(c.`Element ID`)
        WHERE e.onet_soc_code = @soc
          AND c.type = @ctype
        QUALIFY ROW_NUMBER() OVER (PARTITION BY c.`Element Name` ORDER BY e.importance DESC) = 1
        ORDER BY e.importance DESC
        LIMIT 15
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("soc", "STRING", str(onet_code)),
            bigquery.ScalarQueryParameter("ctype", "STRING", ctype),
        ]
    )
    df = bq_client.query(sql, job_config=job_config).to_dataframe()
    if df.empty:
        return f"No {ctype} found for {official_title}."

    bullets = "\n".join(
        [f"- {r['item_name']} (Imp: {r['importance']})" for _, r in df.iterrows()]
    )
    return f"### {ctype} for {official_title}\n\n{bullets}"




# ------------------------------------------------------------------
# TOOL 2: Education - Career Matching & AI Scores
# ------------------------------------------------------------------

    def get_jobs_by_major(tool_context: ToolContext, major_name: str, **kwargs) -> str:
    if not bq_client:
        return "Error: BigQuery not ready."

    program_matches = find_topk_nodes_via_vector(major_name, type="program", k=3)
    if not program_matches:
        return f"Could not find a program match for '{major_name}'."

    tool_context.state["last_major_name"] = program_matches[0]["title"]
    tool_context.state["last_major_matches"] = program_matches

    cip_list = [str(m["id"]) for m in program_matches]
    match_lines = "\n".join([f"- #{m['rank']}: **{m['title']}**" for m in program_matches])

    try:
        sql = f"""
            WITH matched_programs AS (
              SELECT cip_code
              FROM UNNEST(@cip_codes) AS cip_code
            ),
            jobs AS (
              SELECT
                CAST(e.cip_code AS STRING) AS cip_code,
                CAST(e.onet_soc_code AS STRING) AS onet_soc_code,
                o.title AS job_title,
                COALESCE(CAST(s.{AI_SCORE_COL} AS FLOAT64), 0.5) AS ai_score
              FROM `{GCP_PROJECT_ID}.{EDGE_PROGRAM_JOB}` e
              JOIN matched_programs mp
                ON CAST(e.cip_code AS STRING) = mp.cip_code
              JOIN `{GCP_PROJECT_ID}.{NODE_OCCUPATIONS}` o
                ON CAST(e.onet_soc_code AS STRING) = CAST(o.onet_soc_code AS STRING)
              LEFT JOIN `{GCP_PROJECT_ID}.{AI_SCORE_TABLE_ID}` s
                ON CAST(e.onet_soc_code AS STRING) = CAST(s.soc_code AS STRING)
              WHERE o.title IS NOT NULL
            )
            SELECT
              cip_code,
              onet_soc_code,
              job_title,
              ai_score
            FROM jobs
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY cip_code, onet_soc_code, job_title
                ORDER BY ai_score DESC
            ) = 1
        """

        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ArrayQueryParameter("cip_codes", "STRING", cip_list)]
        )

        df = bq_client.query(sql, job_config=job_config).to_dataframe()
        if df.empty:
            return f"No direct career matches found for these program matches:\n{match_lines}"

        df["ai_score"] = pd.to_numeric(df["ai_score"], errors="coerce").fillna(0.5)

        if os.getenv("OBIWAN_DEBUG", "0") == "1":
            print("---- OBIWAN DEBUG: get_jobs_by_major ----")
            print(f"User query major: {major_name}")
            print("Top-3 program matches (debug):")
            for m in program_matches:
                print(f"  #{m['rank']}: {m['title']} (CIP={m['id']}) sim={m['score']:.4f}")
            print(f"Total rows returned: {len(df)}")

            if "onet_soc_code" in df.columns:
                socs = sorted(df["onet_soc_code"].astype(str).unique().tolist())
                print(f"Unique SOC count: {len(socs)}")
                print(f"SOC list: {socs}")

            if {"cip_code", "onet_soc_code"}.issubset(df.columns):
                per_cip = df.groupby("cip_code")["onet_soc_code"].nunique().sort_values(ascending=False)
                print("Unique SOC per CIP:")
                for cip, n in per_cip.items():
                    print(f"  CIP {cip}: {int(n)} SOCs")
            print("------------------------------------------")

        HIGH, LOW = 0.5, 0.2

        jobs_agg = (
            df.groupby("job_title", as_index=False)["ai_score"]
            .max()
            .sort_values("ai_score", ascending=False)
        )

        high_ai = jobs_agg[jobs_agg.ai_score > HIGH].job_title.head(10).tolist()
        safe = (
            jobs_agg[jobs_agg.ai_score < LOW]
            .sort_values("ai_score", ascending=True)
            .job_title.head(10)
            .tolist()
        )
        hybrid = jobs_agg[
            (jobs_agg.ai_score >= LOW) & (jobs_agg.ai_score <= HIGH)
        ].job_title.head(10).tolist()

        out = "### Program Matches (Top 3)\n\n" + match_lines + "\n\n"
        out += "### Career Options (combined from top 3 program matches)\n\n"
        out += "**High AI (>0.5):**\n" + ("\n".join(f"- {t}" for t in high_ai) or "- None") + "\n\n"
        out += "**Safe Haven (<0.2):**\n" + ("\n".join(f"- {t}" for t in safe) or "- None") + "\n\n"
        out += "**Hybrid (0.2–0.5):**\n" + ("\n".join(f"- {t}" for t in hybrid) or "- None")
        return out

    except Exception as e:
        return f"Tool error in get_jobs_by_major: {type(e).__name__}: {e}"




# ------------------------------------------------------------------
# TOOL 3: Institutional Routing & Graph Traversal
# ------------------------------------------------------------------

def _resolve_programs_from_job_via_reverse_edge(
    onet_soc_code: str,
    k: int = 3,
    award_level: Optional[str] = None,
    modality: Optional[str] = None,
    state: Optional[str] = None,
) -> List[Dict[str, object]]:
    """
    Reverse lookup: SOC -> CIPs using EDGE_PROGRAM_JOB, ranked by IPEDS completions (total_completers).
    Filters (award/modality/state) are applied on EDGE_INST_PROGRAM + institutions (NOT on EDGE_PROGRAM_JOB).
    """
    if not bq_client:
        return []

    mod_kw = normalize_modality(modality) if modality else None
    aw_norm = normalize_award_level(award_level) if award_level else None
    award_filters = _award_sql_filters(aw_norm)

    where2 = []
    params = [
        bigquery.ScalarQueryParameter("soc", "STRING", str(onet_soc_code)),
        bigquery.ScalarQueryParameter("k", "INT64", int(k)),
    ]

    if mod_kw:
        where2.append("e.modality = @mod")
        params.append(bigquery.ScalarQueryParameter("mod", "STRING", mod_kw))

    if award_filters:
        where2.append("(" + " OR ".join(award_filters) + ")")

    if state:
        where2.append("LOWER(n.state) = LOWER(@st)")
        params.append(bigquery.ScalarQueryParameter("st", "STRING", state))

    extra_where = (" AND " + " AND ".join(where2)) if where2 else ""

    sql = f"""
      WITH cips_for_soc AS (
        SELECT DISTINCT CAST(cip_code AS STRING) AS cip_code
        FROM `{GCP_PROJECT_ID}.{EDGE_PROGRAM_JOB}` x
        WHERE CAST(x.onet_soc_code AS STRING) = @soc
      ),
      cip_grads AS (
        SELECT
          c.cip_code,
          SUM(COALESCE(e.total_completers, 0)) AS grads
        FROM cips_for_soc c
        JOIN `{GCP_PROJECT_ID}.{EDGE_INST_PROGRAM}` e
          ON CAST(e.cip_code AS STRING) = c.cip_code
        JOIN `{GCP_PROJECT_ID}.{NODE_INSTITUTIONS}` n
          ON CAST(e.institution_id AS STRING) = CAST(n.institution_id AS STRING)
        WHERE 1=1 {extra_where}
        GROUP BY c.cip_code
      )
      SELECT
        g.cip_code,
        p.title AS program_title,
        g.grads
      FROM cip_grads g
      LEFT JOIN `{GCP_PROJECT_ID}.{NODE_PROGRAMS}` p
        ON CAST(p.cip_code AS STRING) = g.cip_code
      WHERE g.grads > 0
      ORDER BY g.grads DESC
      LIMIT @k
    """

    df = bq_client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()

    out: List[Dict[str, object]] = []
    for i, r in df.iterrows():
        title = r["program_title"] if pd.notnull(r["program_title"]) else f"CIP {r['cip_code']}"
        out.append({"id": str(r["cip_code"]), "title": title, "rank": i + 1, "score": None})

    if os.getenv("OBIWAN_DEBUG", "0") == "1":
        print("---- OBIWAN DEBUG: reverse SOC->CIP via completions ----")
        print(f"SOC: {onet_soc_code} | award_level={award_level} | modality={modality} | state={state}")
        for m in out:
            print(f"  #{m['rank']}: {m['title']} (CIP={m['id']})")
        print("--------------------------------------------------------")

    return out


def get_institutions_by_major(
    tool_context: ToolContext,
    major_name: Optional[str] = None,
    award_level: Optional[str] = None,
    modality: Optional[str] = None,
    state: Optional[str] = None,
    **kwargs
) -> str:
    if not bq_client:
        return "Error: BigQuery not initialized."

    resolved_major_name = major_name or tool_context.state.get("last_major_name")

    # Infer program from last job title if no major given
    if not resolved_major_name:
        last_job = tool_context.state.get("last_job_title")
        if last_job:
            soc, official_job = find_node_via_vector(last_job, type="job")

            inferred: List[Dict[str, object]] = []
            if soc:
                inferred = _resolve_programs_from_job_via_reverse_edge(
                    soc, k=3, award_level=award_level, modality=modality, state=state
                )

            # Fallback: vector-search program directly from job title
            if not inferred:
                inferred = find_topk_nodes_via_vector(official_job or last_job, type="program", k=3)

            if inferred:
                tool_context.state["last_major_name"] = inferred[0].get("title") or resolved_major_name
                tool_context.state["last_major_matches"] = inferred
                resolved_major_name = inferred[0].get("title") or resolved_major_name

    # Ask for missing fields ONCE (before any embedding call)
    missing = []
    if not resolved_major_name:
        missing.append("**major/program** (e.g., Counseling, Clinical Mental Health Counseling, Psychology)")
    if not award_level:
        missing.append("**degree level** (Associate, Bachelor, Master, Doctorate, Undergraduate Certificate, Graduate Certificate)")
    if not modality:
        missing.append("**modality** (Online, Hybrid, On-campus)")

    if missing:
        return (
            "To recommend institutions, I need: " + ", ".join(missing) +
            ".\n\nOptional: add a **state** (e.g., CA, WA) to filter results."
        )

    # Top-3 program candidates for CIP lookup
    top3 = find_topk_nodes_via_vector(resolved_major_name, type="program", k=3)

    if os.getenv("OBIWAN_DEBUG", "0") == "1":
        print("---- OBIWAN DEBUG: get_institutions_by_major ----")
        print(f"Resolved major: {resolved_major_name}")
        print("Top-3 program matches (CIP candidates):")
        for i, r in enumerate(top3, 1):
            title = r.get("title")
            cid = r.get("id")
            score = r.get("score")
            score_str = f"{float(score):.4f}" if score is not None else "None"
            print(f"  #{i}: {title} (CIP={cid}) sim={score_str}")
        print(f"Award level: {award_level}")
        print(f"Modality: {modality}")
        print(f"State: {state}")
        print("-----------------------------------------------")

    cip_code = top3[0].get("id") if top3 else None
    official_major = top3[0].get("title") if top3 and top3[0].get("title") else resolved_major_name

    # persist for downstream tools / follow-ups
    tool_context.state["last_major_name"] = official_major
    tool_context.state["last_major_matches"] = top3

    if not cip_code:
        return f"Could not find a program match for '{resolved_major_name}'."

    mod_kw = normalize_modality(modality)
    if not mod_kw:
        return "Modality not recognized. Try: **Online**, **Hybrid**, or **On-campus**."

    aw = normalize_award_level(award_level)
    if aw == "certificate":
        return (
            "When you say **certificate**, do you mean:\n"
            "- **Undergraduate Certificate**...\n"
            "- **Graduate Certificate**...\n"
            "- **Both**\n\n"
            "Reply with: **undergrad certificate**, **graduate certificate**, or **both**."
        )
    if not aw:
        return "Degree level not recognized."

    award_filters = _award_sql_filters(aw)

    where = ["e.modality = @mod"]
    params = [bigquery.ScalarQueryParameter("mod", "STRING", mod_kw)]

    if award_filters:
        where.append("(" + " OR ".join(award_filters) + ")")

    if state:
        where.append("LOWER(n.state) = LOWER(@st)")
        params.append(bigquery.ScalarQueryParameter("st", "STRING", state))

    sql = f"""
        SELECT
          n.name AS institution_name,
          n.state,
          e.award_level,
          e.modality,
          SUM(e.total_completers) AS grads
        FROM `{GCP_PROJECT_ID}.{EDGE_INST_PROGRAM}` e
        JOIN `{GCP_PROJECT_ID}.{NODE_INSTITUTIONS}` n
          ON CAST(e.institution_id AS STRING) = CAST(n.institution_id AS STRING)
        WHERE CAST(e.cip_code AS STRING) = @cip
          AND {" AND ".join(where)}
        GROUP BY institution_name, state, award_level, modality
        HAVING grads > 0
        ORDER BY grads DESC
        LIMIT 10
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=params + [bigquery.ScalarQueryParameter("cip", "STRING", str(cip_code))]
    )

    df = bq_client.query(sql, job_config=job_config).to_dataframe()
    if df.empty:
        return (
            f"No institutions found for **{official_major}** with filters: {award_level} / {modality}"
            + (f" / {state}" if state else "")
        )

    out = (
        f"### Top Schools for {official_major}\n"
        f"Filters: **{award_level}**, **{modality}**" + (f", **{state}**" if state else "") + "\n\n"
    )
    for i, r in df.iterrows():
        out += f"{i+1}. **{r['institution_name']}** ({r['state']})\n"
    return out

# 4. Agent Architecture

In [ ]:
# ------------------------------------------------------------------
# ROOT AGENT
# ------------------------------------------------------------------
root_agent = LlmAgent(
    name="OBI_WAN_Navigator",
    model=Gemini(model="models/gemini-2.5-flash", api_key=GOOGLE_API_KEY),
    instruction="""
You are OBI-WAN (Occupation-Based Index for Workforce AI Navigator).

MANDATORY OUTPUT RULES:
1) For Skills: The tool will provide a list of 15 items. You MUST display the full list. DO NOT SUMMARIZE. Display them as a clean bulleted list.
2) For Jobs:
   - Show "Program Matches (Top 3)" as titles only (no CIP codes, no similarity scores).
   - Then group jobs by:
     - High AI (>0.5)
     - Safe Haven (<0.2)
     - Hybrid (0.2–0.5)
3) Institutions (critical):
   - If the user asks for schools and any of these are missing: major/program, degree level, modality,
     you MUST ask for ALL missing fields in ONE question.
   - If the user previously asked about skills for a specific job/occupation, you may infer the major/program automatically.
   - Use these user-friendly choices:
     - Modality: Online / Hybrid / On-campus
     - Degree level: Associate / Bachelor / Master / Doctorate / Undergraduate Certificate / Graduate Certificate
   - If user says only "certificate", you MUST ask: "Undergraduate Certificate, Graduate Certificate, or Both?"

TONE: Concise, Data-Driven, Strategic. Focus on the data lists.
""",
    tools=[get_job_requirements, get_jobs_by_major, get_institutions_by_major],
)

# 5. Prototype Interface (ADK Web UI)

In [ ]:
from google.colab import output

# Colab uses a different port forwarding method.
# We serve on port 8000.
PORT = 8000

print("Launching OBI-WAN Web Interface...")

# This helper creates a clickable link specifically for your Colab session
output.serve_kernel_port_as_window(PORT)

# Launch the ADK web runner
!adk web --port {PORT} --log_level DEBUG

# 6. Prototype Demonstration

So all together,

In [ ]:
!mkdir career_coach_agent

In [ ]:
%%writefile career_coach_agent/agent.py
import os
import json
from typing import Optional, Tuple, List, Dict

import numpy as np
import pandas as pd


# --- FOR COLAB ---
from google.colab import userdata
from google.colab import auth

from google.cloud import bigquery


import vertexai
from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.tool_context import ToolContext


# ------------------------------------------------------------------
# --- CONFIGURATION ---
# ------------------------------------------------------------------
GCP_PROJECT_ID = "dashboard-458701"
REGION = "us-west1"

JOB_EMBED_TABLE_ID = "onet.job_embeddings_v2"
PROGRAM_EMBED_TABLE_ID = "ipeds.program_embeddings"

NODE_OCCUPATIONS = "onet.occupations_node"
NODE_COMPETENCIES = "onet.competencies_node"
NODE_PROGRAMS = "ipeds.programs_node"
NODE_INSTITUTIONS = "ipeds.institutions_node"

EDGE_JOB_COMPETENCY = "onet.job_requires_competencies_edge"
EDGE_PROGRAM_JOB = "crosswalk.program_leads_to_job_edge"
EDGE_INST_PROGRAM = "ipeds.institution_offers_programs_edge"

AI_SCORE_TABLE_ID = "onet.final_ai_applicability_multiplicative"
AI_SCORE_COL = "AI_Applicability_Score"

EMBED_MODEL_NAME = "text-embedding-004"


# ------------------------------------------------------------------
# --- INIT ---
# ------------------------------------------------------------------
def initialize_services():
    try:
        # 1. Authenticate with Google Cloud (Replaces the GCP_KEY JSON)
        # This triggers a pop-up in Colab asking you to grant access.
        print("⏳ Waiting for Google Cloud authentication popup...")
        auth.authenticate_user()

        # 2. Access the Gemini API Key from Colab's Secrets manager
        google_api_key = userdata.get("GOOGLE_API_KEY")
        os.environ["GOOGLE_API_KEY"] = google_api_key

        # 3. Initialize BigQuery (Automatically uses the auth from step 1)
        bq = bigquery.Client(
            project=GCP_PROJECT_ID,
            location=REGION,
        )

        # 4. Initialize Vertex AI (Automatically uses the auth from step 1)
        vertexai.init(project=GCP_PROJECT_ID, location=REGION)
        emb = TextEmbeddingModel.from_pretrained(EMBED_MODEL_NAME)

        print("BigQuery + Vertex initialized")
        if os.getenv("OBIWAN_DEBUG", "0") == "1":
            print("OBIWAN_DEBUG=1")
        return bq, emb, google_api_key

    except Exception as e:
        print(f"Initialization Error: {e}")
        return None, None, None

# Execute the function so the variables are populated
bq_client, embed_model, GOOGLE_API_KEY = initialize_services()


import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)


# ------------------------------------------------------------------
# --- CACHING GLOBALS ---
# ------------------------------------------------------------------
job_registry_df = None
job_emb_matrix = None

program_registry_df = None
program_emb_matrix = None


# ------------------------------------------------------------------
# 1) VECTOR SEARCH HELPERS
# ------------------------------------------------------------------
def load_job_registry():
    global job_registry_df, job_emb_matrix
    if not bq_client:
        raise RuntimeError("BigQuery client not initialized.")

    if job_registry_df is None:
        print("⏳ Loading Job Vector Registry...")
        sql = f"""
            SELECT soc_code AS onet_soc_code, title AS Title, embedding
            FROM `{GCP_PROJECT_ID}.{JOB_EMBED_TABLE_ID}`
            WHERE embedding IS NOT NULL
        """
        job_registry_df = bq_client.query(sql).to_dataframe()
        job_emb_matrix = np.array(job_registry_df["embedding"].tolist(), dtype=np.float32)

        if job_emb_matrix.size > 0:
            print(
                f"Job Registry Loaded: {len(job_registry_df)} jobs (dim={job_emb_matrix.shape[1]})."
            )
        else:
            print("Job Registry loaded, but embedding matrix is empty.")


def load_program_registry():
    global program_registry_df, program_emb_matrix
    if not bq_client:
        raise RuntimeError("BigQuery client not initialized.")

    if program_registry_df is None:
        print("Loading Program Vector Registry...")
        sql = f"""
            SELECT cip_code, title, embedding
            FROM `{GCP_PROJECT_ID}.{PROGRAM_EMBED_TABLE_ID}`
            WHERE embedding IS NOT NULL
        """
        program_registry_df = bq_client.query(sql).to_dataframe()
        program_emb_matrix = np.array(
            program_registry_df["embedding"].tolist(), dtype=np.float32
        )

        if program_emb_matrix.size > 0:
            print(
                f"Program Registry Loaded: {len(program_registry_df)} programs (dim={program_emb_matrix.shape[1]})."
            )
        else:
            print("Program Registry loaded, but embedding matrix is empty.")


def _embed_query_vertex(user_query: str) -> np.ndarray:
    if embed_model is None:
        raise RuntimeError("Embedding model not initialized.")
    if user_query is None or str(user_query).strip() == "":
        raise ValueError("User query is empty; cannot embed.")

    inputs = [TextEmbeddingInput(str(user_query), task_type="RETRIEVAL_QUERY")]
    result = embed_model.get_embeddings(inputs)
    return np.array(result[0].values, dtype=np.float32)


def _cosine_sims(matrix: np.ndarray, q_vec: np.ndarray) -> np.ndarray:
    denom = (np.linalg.norm(matrix, axis=1) * np.linalg.norm(q_vec) + 1e-8)
    return np.dot(matrix, q_vec) / denom


def find_topk_nodes_via_vector(
    user_query: str, type: str = "program", k: int = 3
) -> List[Dict[str, object]]:
    if type == "job":
        load_job_registry()
        df, matrix, id_col, title_col = (
            job_registry_df,
            job_emb_matrix,
            "onet_soc_code",
            "Title",
        )
    else:
        load_program_registry()
        df, matrix, id_col, title_col = (
            program_registry_df,
            program_emb_matrix,
            "cip_code",
            "title",
        )

    if df is None or df.empty or matrix is None or matrix.size == 0:
        return []

    q_vec = _embed_query_vertex(user_query)

    if matrix.shape[1] != q_vec.shape[0]:
        raise ValueError(
            f"Embedding dimension mismatch for type='{type}'. "
            f"Registry dim={matrix.shape[1]}, query dim={q_vec.shape[0]}."
        )

    sims = _cosine_sims(matrix, q_vec)
    k = max(1, int(k))
    k = min(k, len(sims))

    top_idx = np.argpartition(-sims, kth=k - 1)[:k]
    top_idx = top_idx[np.argsort(-sims[top_idx])]

    out: List[Dict[str, object]] = []
    for rank, idx in enumerate(top_idx, start=1):
        row = df.iloc[int(idx)]
        out.append(
            {
                "id": row[id_col],
                "title": row[title_col],
                "score": float(sims[int(idx)]),
                "rank": rank,
            }
        )

    if os.getenv("OBIWAN_DEBUG", "0") == "1":
        print(f"---- OBIWAN DEBUG: find_topk_nodes_via_vector(type={type}) ----")
        print(f"Query: {user_query}")
        for m in out:
            print(
                f"  #{m['rank']}: {m['title']} (ID={m['id']}) score={m['score']:.4f}"
            )
        print("--------------------------------------------------------------")

    return out


def find_node_via_vector(user_query: str, type: str = "job") -> Tuple[Optional[str], Optional[str]]:
    matches = find_topk_nodes_via_vector(user_query, type=type, k=1)
    if not matches:
        return None, None
    return str(matches[0]["id"]), str(matches[0]["title"])


# ------------------------------------------------------------------
# 2) NORMALIZERS
# ------------------------------------------------------------------
def normalize_comp_category(category: Optional[str]) -> str:
    c = (category or "Skill").strip().lower()
    if "know" in c:
        return "Knowledge"
    if "abil" in c:
        return "Ability"
    return "Skill"


def normalize_award_level(user_text: Optional[str]) -> Optional[str]:
    if not user_text:
        return None
    t = user_text.strip().lower()
    if "associate" in t:
        return "associate"
    if "bachelor" in t or "undergrad degree" in t:
        return "bachelor"
    if "master" in t:
        return "master"
    if "doctor" in t or "phd" in t:
        return "doctor"
    if "graduate certificate" in t or "grad certificate" in t:
        return "grad_cert"
    if "undergraduate certificate" in t or "undergrad certificate" in t:
        return "ug_cert"
    if "certificate" in t:
        return "certificate"
    return None


def normalize_modality(user_text: Optional[str]) -> Optional[str]:
    if not user_text:
        return None
    t = user_text.strip().lower()
    if "online" in t or "remote" in t or "virtual" in t:
        return "DE ENTIRELY"
    if "hybrid" in t or "some" in t or "mixed" in t:
        return "DE SOME"
    if "on-campus" in t or "campus" in t or "person" in t or "f2f" in t:
        return "F2F"
    return None


def _award_sql_filters(aw_norm: Optional[str]) -> List[str]:
    if not aw_norm:
        return []
    if aw_norm == "associate":
        return ['e.award_level = "Associate\'s degree"']
    if aw_norm == "bachelor":
        return ['e.award_level = "Bachelor"']
    if aw_norm == "master":
        return ['e.award_level = "Master\'s degree"']
    if aw_norm == "doctor":
        return ["e.award_level LIKE 'Doctor%'", "e.award_level LIKE 'Doctoral%'"]
    if aw_norm == "ug_cert":
        return ["e.award_level LIKE 'Certificates of%'"]
    if aw_norm == "grad_cert":
        return [
            'e.award_level = "Post-master\'s certificate"',
            'e.award_level = "Postbaccalaureate certificate"',
        ]
    return []


# ------------------------------------------------------------------
# 3) TOOLS
# ------------------------------------------------------------------
def get_job_requirements(
    tool_context: ToolContext, job_title: str, category: str = "Skill", **kwargs
) -> str:
    if not bq_client:
        return "Error: BigQuery not ready."

    onet_code, official_title = find_node_via_vector(job_title, type="job")
    if not onet_code:
        return f"Could not find a match for '{job_title}'."

    tool_context.state["last_job_title"] = official_title
    ctype = normalize_comp_category(category)

    sql = f"""
        SELECT c.`Element Name` AS item_name, e.importance
        FROM `{GCP_PROJECT_ID}.{EDGE_JOB_COMPETENCY}` e
        JOIN `{GCP_PROJECT_ID}.{NODE_COMPETENCIES}` c
          ON TRIM(e.element_id) = TRIM(c.`Element ID`)
        WHERE e.onet_soc_code = @soc
          AND c.type = @ctype
        QUALIFY ROW_NUMBER() OVER (PARTITION BY c.`Element Name` ORDER BY e.importance DESC) = 1
        ORDER BY e.importance DESC
        LIMIT 15
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("soc", "STRING", str(onet_code)),
            bigquery.ScalarQueryParameter("ctype", "STRING", ctype),
        ]
    )
    df = bq_client.query(sql, job_config=job_config).to_dataframe()
    if df.empty:
        return f"No {ctype} found for {official_title}."

    bullets = "\n".join(
        [f"- {r['item_name']} (Imp: {r['importance']})" for _, r in df.iterrows()]
    )
    return f"### {ctype} for {official_title}\n\n{bullets}"


def get_jobs_by_major(tool_context: ToolContext, major_name: str, **kwargs) -> str:
    if not bq_client:
        return "Error: BigQuery not ready."

    program_matches = find_topk_nodes_via_vector(major_name, type="program", k=3)
    if not program_matches:
        return f"Could not find a program match for '{major_name}'."

    tool_context.state["last_major_name"] = program_matches[0]["title"]
    tool_context.state["last_major_matches"] = program_matches

    cip_list = [str(m["id"]) for m in program_matches]
    match_lines = "\n".join([f"- #{m['rank']}: **{m['title']}**" for m in program_matches])

    try:
        sql = f"""
            WITH matched_programs AS (
              SELECT cip_code
              FROM UNNEST(@cip_codes) AS cip_code
            ),
            jobs AS (
              SELECT
                CAST(e.cip_code AS STRING) AS cip_code,
                CAST(e.onet_soc_code AS STRING) AS onet_soc_code,
                o.title AS job_title,
                COALESCE(CAST(s.{AI_SCORE_COL} AS FLOAT64), 0.5) AS ai_score
              FROM `{GCP_PROJECT_ID}.{EDGE_PROGRAM_JOB}` e
              JOIN matched_programs mp
                ON CAST(e.cip_code AS STRING) = mp.cip_code
              JOIN `{GCP_PROJECT_ID}.{NODE_OCCUPATIONS}` o
                ON CAST(e.onet_soc_code AS STRING) = CAST(o.onet_soc_code AS STRING)
              LEFT JOIN `{GCP_PROJECT_ID}.{AI_SCORE_TABLE_ID}` s
                ON CAST(e.onet_soc_code AS STRING) = CAST(s.soc_code AS STRING)
              WHERE o.title IS NOT NULL
            )
            SELECT
              cip_code,
              onet_soc_code,
              job_title,
              ai_score
            FROM jobs
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY cip_code, onet_soc_code, job_title
                ORDER BY ai_score DESC
            ) = 1
        """

        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ArrayQueryParameter("cip_codes", "STRING", cip_list)]
        )

        df = bq_client.query(sql, job_config=job_config).to_dataframe()
        if df.empty:
            return f"No direct career matches found for these program matches:\n{match_lines}"

        df["ai_score"] = pd.to_numeric(df["ai_score"], errors="coerce").fillna(0.5)

        if os.getenv("OBIWAN_DEBUG", "0") == "1":
            print("---- OBIWAN DEBUG: get_jobs_by_major ----")
            print(f"User query major: {major_name}")
            print("Top-3 program matches (debug):")
            for m in program_matches:
                print(f"  #{m['rank']}: {m['title']} (CIP={m['id']}) sim={m['score']:.4f}")
            print(f"Total rows returned: {len(df)}")

            if "onet_soc_code" in df.columns:
                socs = sorted(df["onet_soc_code"].astype(str).unique().tolist())
                print(f"Unique SOC count: {len(socs)}")
                print(f"SOC list: {socs}")

            if {"cip_code", "onet_soc_code"}.issubset(df.columns):
                per_cip = df.groupby("cip_code")["onet_soc_code"].nunique().sort_values(ascending=False)
                print("Unique SOC per CIP:")
                for cip, n in per_cip.items():
                    print(f"  CIP {cip}: {int(n)} SOCs")
            print("------------------------------------------")

        HIGH, LOW = 0.5, 0.2

        jobs_agg = (
            df.groupby("job_title", as_index=False)["ai_score"]
            .max()
            .sort_values("ai_score", ascending=False)
        )

        high_ai = jobs_agg[jobs_agg.ai_score > HIGH].job_title.head(10).tolist()
        safe = (
            jobs_agg[jobs_agg.ai_score < LOW]
            .sort_values("ai_score", ascending=True)
            .job_title.head(10)
            .tolist()
        )
        hybrid = jobs_agg[
            (jobs_agg.ai_score >= LOW) & (jobs_agg.ai_score <= HIGH)
        ].job_title.head(10).tolist()

        out = "### Program Matches (Top 3)\n\n" + match_lines + "\n\n"
        out += "### Career Options (combined from top 3 program matches)\n\n"
        out += "**High AI (>0.5):**\n" + ("\n".join(f"- {t}" for t in high_ai) or "- None") + "\n\n"
        out += "**Safe Haven (<0.2):**\n" + ("\n".join(f"- {t}" for t in safe) or "- None") + "\n\n"
        out += "**Hybrid (0.2–0.5):**\n" + ("\n".join(f"- {t}" for t in hybrid) or "- None")
        return out

    except Exception as e:
        return f"Tool error in get_jobs_by_major: {type(e).__name__}: {e}"


def _resolve_programs_from_job_via_reverse_edge(
    onet_soc_code: str,
    k: int = 3,
    award_level: Optional[str] = None,
    modality: Optional[str] = None,
    state: Optional[str] = None,
) -> List[Dict[str, object]]:
    """
    Reverse lookup: SOC -> CIPs using EDGE_PROGRAM_JOB, ranked by IPEDS completions (total_completers).
    Filters (award/modality/state) are applied on EDGE_INST_PROGRAM + institutions (NOT on EDGE_PROGRAM_JOB).
    """
    if not bq_client:
        return []

    mod_kw = normalize_modality(modality) if modality else None
    aw_norm = normalize_award_level(award_level) if award_level else None
    award_filters = _award_sql_filters(aw_norm)

    where2 = []
    params = [
        bigquery.ScalarQueryParameter("soc", "STRING", str(onet_soc_code)),
        bigquery.ScalarQueryParameter("k", "INT64", int(k)),
    ]

    if mod_kw:
        where2.append("e.modality = @mod")
        params.append(bigquery.ScalarQueryParameter("mod", "STRING", mod_kw))

    if award_filters:
        where2.append("(" + " OR ".join(award_filters) + ")")

    if state:
        where2.append("LOWER(n.state) = LOWER(@st)")
        params.append(bigquery.ScalarQueryParameter("st", "STRING", state))

    extra_where = (" AND " + " AND ".join(where2)) if where2 else ""

    sql = f"""
      WITH cips_for_soc AS (
        SELECT DISTINCT CAST(cip_code AS STRING) AS cip_code
        FROM `{GCP_PROJECT_ID}.{EDGE_PROGRAM_JOB}` x
        WHERE CAST(x.onet_soc_code AS STRING) = @soc
      ),
      cip_grads AS (
        SELECT
          c.cip_code,
          SUM(COALESCE(e.total_completers, 0)) AS grads
        FROM cips_for_soc c
        JOIN `{GCP_PROJECT_ID}.{EDGE_INST_PROGRAM}` e
          ON CAST(e.cip_code AS STRING) = c.cip_code
        JOIN `{GCP_PROJECT_ID}.{NODE_INSTITUTIONS}` n
          ON CAST(e.institution_id AS STRING) = CAST(n.institution_id AS STRING)
        WHERE 1=1 {extra_where}
        GROUP BY c.cip_code
      )
      SELECT
        g.cip_code,
        p.title AS program_title,
        g.grads
      FROM cip_grads g
      LEFT JOIN `{GCP_PROJECT_ID}.{NODE_PROGRAMS}` p
        ON CAST(p.cip_code AS STRING) = g.cip_code
      WHERE g.grads > 0
      ORDER BY g.grads DESC
      LIMIT @k
    """

    df = bq_client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()

    out: List[Dict[str, object]] = []
    for i, r in df.iterrows():
        title = r["program_title"] if pd.notnull(r["program_title"]) else f"CIP {r['cip_code']}"
        out.append({"id": str(r["cip_code"]), "title": title, "rank": i + 1, "score": None})

    if os.getenv("OBIWAN_DEBUG", "0") == "1":
        print("---- OBIWAN DEBUG: reverse SOC->CIP via completions ----")
        print(f"SOC: {onet_soc_code} | award_level={award_level} | modality={modality} | state={state}")
        for m in out:
            print(f"  #{m['rank']}: {m['title']} (CIP={m['id']})")
        print("--------------------------------------------------------")

    return out


def get_institutions_by_major(
    tool_context: ToolContext,
    major_name: Optional[str] = None,
    award_level: Optional[str] = None,
    modality: Optional[str] = None,
    state: Optional[str] = None,
    **kwargs
) -> str:
    if not bq_client:
        return "Error: BigQuery not initialized."

    resolved_major_name = major_name or tool_context.state.get("last_major_name")

    # Infer program from last job title if no major given
    if not resolved_major_name:
        last_job = tool_context.state.get("last_job_title")
        if last_job:
            soc, official_job = find_node_via_vector(last_job, type="job")

            inferred: List[Dict[str, object]] = []
            if soc:
                inferred = _resolve_programs_from_job_via_reverse_edge(
                    soc, k=3, award_level=award_level, modality=modality, state=state
                )

            # Fallback: vector-search program directly from job title
            if not inferred:
                inferred = find_topk_nodes_via_vector(official_job or last_job, type="program", k=3)

            if inferred:
                tool_context.state["last_major_name"] = inferred[0].get("title") or resolved_major_name
                tool_context.state["last_major_matches"] = inferred
                resolved_major_name = inferred[0].get("title") or resolved_major_name

    # Ask for missing fields ONCE (before any embedding call)
    missing = []
    if not resolved_major_name:
        missing.append("**major/program** (e.g., Counseling, Clinical Mental Health Counseling, Psychology)")
    if not award_level:
        missing.append("**degree level** (Associate, Bachelor, Master, Doctorate, Undergraduate Certificate, Graduate Certificate)")
    if not modality:
        missing.append("**modality** (Online, Hybrid, On-campus)")

    if missing:
        return (
            "To recommend institutions, I need: " + ", ".join(missing) +
            ".\n\nOptional: add a **state** (e.g., CA, WA) to filter results."
        )

    # Top-3 program candidates for CIP lookup
    top3 = find_topk_nodes_via_vector(resolved_major_name, type="program", k=3)

    if os.getenv("OBIWAN_DEBUG", "0") == "1":
        print("---- OBIWAN DEBUG: get_institutions_by_major ----")
        print(f"Resolved major: {resolved_major_name}")
        print("Top-3 program matches (CIP candidates):")
        for i, r in enumerate(top3, 1):
            title = r.get("title")
            cid = r.get("id")
            score = r.get("score")
            score_str = f"{float(score):.4f}" if score is not None else "None"
            print(f"  #{i}: {title} (CIP={cid}) sim={score_str}")
        print(f"Award level: {award_level}")
        print(f"Modality: {modality}")
        print(f"State: {state}")
        print("-----------------------------------------------")

    cip_code = top3[0].get("id") if top3 else None
    official_major = top3[0].get("title") if top3 and top3[0].get("title") else resolved_major_name

    # persist for downstream tools / follow-ups
    tool_context.state["last_major_name"] = official_major
    tool_context.state["last_major_matches"] = top3

    if not cip_code:
        return f"Could not find a program match for '{resolved_major_name}'."

    mod_kw = normalize_modality(modality)
    if not mod_kw:
        return "Modality not recognized. Try: **Online**, **Hybrid**, or **On-campus**."

    aw = normalize_award_level(award_level)
    if aw == "certificate":
        return (
            "When you say **certificate**, do you mean:\n"
            "- **Undergraduate Certificate**...\n"
            "- **Graduate Certificate**...\n"
            "- **Both**\n\n"
            "Reply with: **undergrad certificate**, **graduate certificate**, or **both**."
        )
    if not aw:
        return "Degree level not recognized."

    award_filters = _award_sql_filters(aw)

    where = ["e.modality = @mod"]
    params = [bigquery.ScalarQueryParameter("mod", "STRING", mod_kw)]

    if award_filters:
        where.append("(" + " OR ".join(award_filters) + ")")

    if state:
        where.append("LOWER(n.state) = LOWER(@st)")
        params.append(bigquery.ScalarQueryParameter("st", "STRING", state))

    sql = f"""
        SELECT
          n.name AS institution_name,
          n.state,
          e.award_level,
          e.modality,
          SUM(e.total_completers) AS grads
        FROM `{GCP_PROJECT_ID}.{EDGE_INST_PROGRAM}` e
        JOIN `{GCP_PROJECT_ID}.{NODE_INSTITUTIONS}` n
          ON CAST(e.institution_id AS STRING) = CAST(n.institution_id AS STRING)
        WHERE CAST(e.cip_code AS STRING) = @cip
          AND {" AND ".join(where)}
        GROUP BY institution_name, state, award_level, modality
        HAVING grads > 0
        ORDER BY grads DESC
        LIMIT 10
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=params + [bigquery.ScalarQueryParameter("cip", "STRING", str(cip_code))]
    )

    df = bq_client.query(sql, job_config=job_config).to_dataframe()
    if df.empty:
        return (
            f"No institutions found for **{official_major}** with filters: {award_level} / {modality}"
            + (f" / {state}" if state else "")
        )

    out = (
        f"### Top Schools for {official_major}\n"
        f"Filters: **{award_level}**, **{modality}**" + (f", **{state}**" if state else "") + "\n\n"
    )
    for i, r in df.iterrows():
        out += f"{i+1}. **{r['institution_name']}** ({r['state']})\n"
    return out


# ------------------------------------------------------------------
# 4) ROOT AGENT
# ------------------------------------------------------------------
root_agent = LlmAgent(
    name="OBI_WAN_Navigator",
    model=Gemini(model="models/gemini-2.5-flash", api_key=GOOGLE_API_KEY),
    instruction="""
You are OBI-WAN (Occupation-Based Index for Workforce AI Navigator).

MANDATORY OUTPUT RULES:
1) For Skills: The tool will provide a list of 15 items. You MUST display the full list. DO NOT SUMMARIZE. Display them as a clean bulleted list.
2) For Jobs:
   - Show "Program Matches (Top 3)" as titles only (no CIP codes, no similarity scores).
   - Then group jobs by:
     - High AI (>0.5)
     - Safe Haven (<0.2)
     - Hybrid (0.2–0.5)
3) Institutions (critical):
   - If the user asks for schools and any of these are missing: major/program, degree level, modality,
     you MUST ask for ALL missing fields in ONE question.
   - If the user previously asked about skills for a specific job/occupation, you may infer the major/program automatically.
   - Use these user-friendly choices:
     - Modality: Online / Hybrid / On-campus
     - Degree level: Associate / Bachelor / Master / Doctorate / Undergraduate Certificate / Graduate Certificate
   - If user says only "certificate", you MUST ask: "Undergraduate Certificate, Graduate Certificate, or Both?"

TONE: Concise, Data-Driven, Strategic. Focus on the data lists.
""",
    tools=[get_job_requirements, get_jobs_by_major, get_institutions_by_major],
)




Writing career_coach_agent/agent.py


In [ ]:
from google.colab import output

# Colab uses a different port forwarding method.
# We serve on port 8000.
PORT = 8000

print("Launching OBI-WAN Web Interface...")

# This helper creates a clickable link specifically for your Colab session
output.serve_kernel_port_as_window(PORT)

# Launch the ADK web runner
!adk web --port {PORT} --log_level DEBUG

The demonstration of the agent is shown in the accompanying YouTube video!

https://youtu.be/ULZau9wDRtA

# 7. Adversarial Safety Evaluation